# Track B, advanced task in 2D: the x axis by reinforcement learning

Same steps as `track_b.ipynb`, for a second policy. The height keeps the Track B policy (`rl_controller_platform.m`);
this policy decides the **difference δ between motors 3 and 1** (u1 = u_z − δ, u3 = u_z + δ), which tilts the drone and moves
it along x. It replaces the cascade of `platform_controller.m` (position loop + angle loop): it receives the gap to the
platform of the vehicle and the angle, and sends δ directly.

## 1. The environment (`drone_env_x.py`, built like `drone_env.py`)

- **Model:** the pitch axis identified in `x_axis.ipynb` from `explore_x.csv`: motors of the 1st order (τ ≈ 0.2 s),
  F = tanh(0.992 x_m), dω_y/dt = 100 (F3 − F1), dθ/dt = ω_y, dvx/dt = g tan θ.
- **Observation (6 numbers):** gap of position to the vehicle, gap of speed, θ / 0.2, ω_y, previous action, and the
  acceleration of the vehicle written as the angle it needs (a / g / 0.2: in a curve the drone must already be tilted).
  θ is the estimate of the complementary filter (the measured θ is unusable in `UAV_model.slx`).
- **Action:** a ∈ [−1, 1], δ = 0.1 a (same bound as the cascade).
- **Reward** (`compute_reward`): −|gap| + precision bonus 0.5 exp(−(gap/0.05)²) − 0.1 (speed gap)²
  − W_JUMP (a − a_prev)² − W_ZIGZAG (a − 2 a_prev + a_prev2)² (smoothing, the same two terms as the height policy)
  − 2 × (|θ| − 0.2 if |θ| > 0.2), and −100 if the drone is more than 3 m away or flips (|θ| > 0.8 rad).
  First try with only 0.05 (a − a_prev)²: fast but δ jumped from one bound to the other (bang-bang), risky on the real drone.
- **Training mode** (`train=True`): random scenarios (the vehicle starts at a random time and speed, or the target jumps
  up to ±1.2 m: "land elsewhere", or the vehicle drives a curve, seen along or across the axis), noise, gain of the pitch
  ±30 %, thrust ±10 %, common command u_z between 0.2 and 0.6.
- **3D:** the roll axis (y, motors 2-4) has the same dynamics, so the SAME policy flies y in `platform3d_rl_controller.m`
  (with y' = −y, because φ > 0 moves the drone towards −y).
- **Evaluation mode** (default): the scenario of `car_position.m` (0.3 m/s from 1.5 s), no noise.

The cell below also flies the cascade (`cascade_policy`) in this environment: it is the reference to beat
(on the real drone it gave a gap of ~1 cm at the contact).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from drone_env_x import DroneEnvX, run_episode, indicators, cascade_policy

check_env(DroneEnvX())
print('check_env: OK')

SCENARIOS = {'vehicle 0.3 m/s (car_position.m)': None,
             'land 1 m further': {'v': 0.0, 'jump': 1.0},
             'vehicle 0.6 m/s': {'v': 0.6},
             'curve R = 1.5 m, along x (car_path.m)': {'R': 1.5, 'axis': 0},
             'curve R = 1.5 m, along y (car_path.m)': {'R': 1.5, 'axis': 1}}
for name, opt in SCENARIOS.items():
    L, R = run_episode(DroneEnvX(), cascade_policy, options=opt)
    print(f'cascade, {name:34s} return {R:6.1f}  ', {k: round(float(v), 3) for k, v in indicators(L).items()})

## 2. Training and export

Same cell as `track_b.ipynb` (SAC, `MlpPolicy` 64 × 64, `EvalCallback` that keeps the best agent, `seed=0`).
The model of the x axis is much cheaper than the RNN, but with the smoothing terms 30 000 steps are not enough
(tested: smooth but 8 cm behind the vehicle): **100 000 steps, ≈ 20-30 min on a laptop.**

The export: `export_for_matlab` of `export_policy.py` is made for the height (it needs z, vz… in `info["m"]`), so we only use
its function that converts the network into matrices (`_policy_matrices`, file not edited) and write `policy_x.mat`.
We add observations and the actions computed by Python (`test_obs`, `test_act`): at its first call, `rl_controller_x.m`
recomputes them and stops if MATLAB and Python disagree.

In [ ]:
from scipy.io import savemat
from stable_baselines3 import SAC
from export_policy import _policy_matrices

env = DroneEnvX()                                                          # evaluation scenario, no noise
train_env = Monitor(DroneEnvX(train=True, seed=0), '../results/checkpoints/sac_x/')
best = EvalCallback(Monitor(DroneEnvX()), eval_freq=5_000, n_eval_episodes=1,
                    best_model_save_path='../results/checkpoints/best_x/', deterministic=True, verbose=1)

agent = SAC("MlpPolicy", train_env, verbose=0, seed=0, policy_kwargs=dict(net_arch=[64, 64]))
agent.learn(total_timesteps=100_000, callback=best)
agent = SAC.load('../results/checkpoints/best_x/best_model')               # the best agent seen during training

# Export: matrices of the network + test observations (noisy training scenarios: varied inputs)
test_env, obs_list = DroneEnvX(train=True, seed=123), []
for seed in range(5):
    obs, _ = test_env.reset(seed=seed)
    for _ in range(20):
        obs_list.append(obs)
        obs, *_ = test_env.step(agent.predict(obs, deterministic=True)[0])
test_obs = np.array(obs_list, dtype=np.float64)
test_act = np.array([agent.predict(o, deterministic=True)[0][0] for o in obs_list], dtype=np.float64).reshape(-1, 1)
savemat('policy_x.mat', {**_policy_matrices(agent), 'test_obs': test_obs, 'test_act': test_act})
print('policy_x.mat written')

## 3. Check on the model before MATLAB

Three scenarios, RL policy against the cascade. Targets before MATLAB: gap at 5.5 s (start of the landing) of a few cm,
no oscillation, |θ| below 0.2 rad, small zigzag of δ.

In [ ]:
policy = lambda obs: agent.predict(obs, deterministic=True)[0]
fig, ax = plt.subplots(len(SCENARIOS), 2, figsize=(12, 13), sharex=True)
for i, (name, opt) in enumerate(SCENARIOS.items()):
    for label, pol, col in [('RL', policy, 'r'), ('cascade', cascade_policy, 'tab:blue')]:
        L, R = run_episode(DroneEnvX(), pol, options=opt)
        print(f'{label:8s} {name:34s} return {R:6.1f}  ', {k: round(float(v), 3) for k, v in indicators(L).items()})
        t, x, xc, th, d = L.T
        ax[i, 0].plot(t, x - xc, col, label=label); ax[i, 1].step(t, d, col, where='post', label=label)
    ax[i, 0].set_ylabel('x - x_vehicle [m]'); ax[i, 0].set_title(name); ax[i, 1].set_ylabel('delta'); ax[i, 1].set_title(name)
    ax[i, 0].grid(); ax[i, 1].grid(); ax[i, 0].legend()
ax[-1, 0].set_xlabel('t [s]'); ax[-1, 1].set_xlabel('t [s]'); plt.tight_layout(); plt.show()

## 4. In MATLAB

The cell below copies `policy_x.mat` into the `rl_interface` folder. Then in MATLAB (current folder = `rl_interface`):
- **2D** (vehicle in a straight line): `EINT_PLATFORM_RL_eval` → `platform_rl_flight.csv`;
- **3D** (vehicle in a curve): `EINT_PLATFORM3D_eval` (cascade, if not done yet) and `EINT_PLATFORM3D_RL_eval` (RL) →
  `platform3d_flight.csv` and `platform3d_rl_flight.csv`, compared by `compare_rl_cascade.ipynb`.

Each prints `Check x policy OK` (MATLAB gives the same actions as Python), then flies on `UAV_model.slx`.

In [ ]:
import shutil
from pathlib import Path
RL_DIR = Path(r'C:\Users\ezabl\Downloads\Student_folder\Student_folder\rl_interface')   # change if needed
shutil.copy('policy_x.mat', RL_DIR / 'policy_x.mat')
print('copied to', RL_DIR)